# AIDA × Hope: breast ultrasound AI, start to finish

An educational demo for a breast cancer awareness booth. **Not a medical device.**

**How to run (Google Colab)**
1. Put the BUSI dataset `.zip` in Google Drive under `MyDrive/HopeFair/raw/`
   (the original Kaggle BUSI zip or the Curated BUSI zip; both work).
2. Runtime → Change runtime type → **T4 GPU**.
3. Runtime → **Run all**. If Colab disconnects, just Run all again: training resumes from the last epoch.

Everything the notebook produces is saved to `MyDrive/HopeFair/outputs/`:

| Section | Output (in `outputs/`) |
|---|---|
| 1. Load + clean | — |
| 2. Split | `split.csv` |
| 3. Train ResNet18 | `checkpoints/last.pt` (every epoch), `models/busi_resnet18.pt` (best), `training_curves.png` |
| 4. Evaluate (test only) | `metrics.json`, `confusion_matrix.png` |
| 5. Grad-CAM | `gradcam/heatmaps/`, `gradcam/overlays/` |
| 6. Predictions | `predictions.csv` |
| 7. Booth set | `export/` (images, heatmaps, overlays, print, `booth_set.json`, `failure_candidates.csv`) |
| 8. Wisconsin model | `models/wdbc_model.pkl`, `models/wdbc_features.json` |

## Setup: Google Drive, packages, GPU check

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%pip install -q grad-cam imagehash timm

import hashlib
import json
import math
import os
import pickle
import random
import re
import shutil
import time
import zipfile
from collections import defaultdict
from itertools import zip_longest
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Colab choose Runtime > Change runtime type > T4 GPU, "
                       "then Runtime > Run all again.")
print("GPU:", torch.cuda.get_device_name(0))

## Config: every path and setting in one place

In [ ]:
# ---- Paths ----
DRIVE_DIR = Path("/content/drive/MyDrive/HopeFair")
RAW_DIR = DRIVE_DIR / "raw"           # the dataset .zip lives here
ZIP_NAME = None                       # None = the only .zip in RAW_DIR; or e.g. "Dataset_BUSI_with_GT.zip"
DATA_DIR = Path("/content/data")      # unzipped to Colab's local disk (much faster than reading from Drive)
OUT_DIR = DRIVE_DIR / "outputs"       # EVERYTHING we produce is saved here
CKPT_DIR = OUT_DIR / "checkpoints"    # last.pt, overwritten every epoch (for resuming)
MODEL_DIR = OUT_DIR / "models"        # busi_resnet18.pt (best epoch), wdbc_model.pkl, wdbc_features.json
GRADCAM_DIR = OUT_DIR / "gradcam"     # Grad-CAM images for every test image
EXPORT_DIR = OUT_DIR / "export"       # the 30-case booth set
for folder in (DATA_DIR, OUT_DIR, CKPT_DIR, MODEL_DIR, GRADCAM_DIR, EXPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# ---- Dataset layout ----
# "auto"    : detect it
# "kaggle"  : original BUSI: normal/ benign/ malignant/ with "benign (1).png", "benign (1)_mask.png", "_mask_1" ...
# "curated" : Curated BUSI: images/benign_id_1.png + masks/benign_id_1_mask.png
LAYOUT = "auto"

# ---- Training ----
SEED = 42
CLASSES = ["normal", "benign", "malignant"]   # class index 0, 1, 2 everywhere below
IMG_SIZE = 224
EPOCHS = 25
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
RESUME = True                         # continue from checkpoints/last.pt if it exists

# ---- Cleaning, Grad-CAM, booth ----
PHASH_MAX_DISTANCE = 4                # pHash Hamming distance that counts as "the same image"
TOP_FRACTION = 0.20                   # cam_mask_overlap uses the hottest 20% of Grad-CAM pixels
CONTOUR_RGB = (255, 0, 255)           # magenta tumour outline: not part of the blue-to-red heatmap colours
BOOTH_PER_CLASS = 10                  # 10 normal + 10 benign + 10 malignant = 30 cases
MAX_ERRORS = 6                        # false negatives + false positives to include
N_WRONG_REASON = 2                    # "right answer, wrong reason" cases
PRINT_WIDTH = 1748                    # A5 width at 300 dpi (148 mm)

# ---- Wisconsin (WDBC) model ----
WDBC_FEATURES = ["mean radius", "mean texture", "mean smoothness", "mean concavity", "worst concave points"]


def seed_everything(seed):
    """Fix every random number generator so a re-run gives the same split and the same model."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
device = torch.device("cuda")
print("Outputs will be saved to:", OUT_DIR)

## 1. Load + clean

* Unzip the dataset from Drive to the fast local disk (only once per Colab session).
* Detect the folder layout and pair every image with its tumour outline (mask). Images with several
  tumours have several mask files (`_mask`, `_mask_1`, …); we merge them into one. Masks are
  **never** used as model inputs.
* **Near-duplicates:** BUSI contains some scans saved twice. If one copy is in training and the other
  in testing, the test score is inflated. We compare a *perceptual hash* (pHash) of every pair of
  images; a distance ≤ 4 (out of 64 bits) means "practically the same picture". Duplicates are
  grouped and each group will stay inside one split.

In [ ]:
# ---- Unzip (Drive -> local disk) ----
zips = [RAW_DIR / ZIP_NAME] if ZIP_NAME else sorted(RAW_DIR.glob("*.zip"))
if len(zips) != 1 or not zips[0].exists():
    raise FileNotFoundError(f"Expected exactly one dataset .zip in {RAW_DIR}, found {[z.name for z in zips]}. "
                            "Upload it there, or set ZIP_NAME in the Config cell.")
zip_path = zips[0]
unzip_dir = DATA_DIR / zip_path.stem
if unzip_dir.exists():
    print(f"Already unzipped: {unzip_dir}")
else:
    print(f"Unzipping {zip_path.name} -> {unzip_dir} ...")
    partial = DATA_DIR / (zip_path.stem + "_partial")    # so a half-finished unzip is never mistaken for done
    shutil.rmtree(partial, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(partial)
    partial.rename(unzip_dir)
    print("Unzip done.")


# ---- Detect the layout ----
def detect_layout(root, wanted="auto"):
    """Find the folder holding the data: ("curated", folder with images/ + masks/)
    or ("kaggle", folder with normal/ benign/ malignant/)."""
    folders = [root] + sorted(p for p in root.rglob("*") if p.is_dir() and "__MACOSX" not in p.parts)
    for folder in folders:
        if wanted in ("auto", "curated") and (folder / "images").is_dir() and (folder / "masks").is_dir():
            return "curated", folder
        if wanted in ("auto", "kaggle") and all((folder / c).is_dir() for c in CLASSES):
            return "kaggle", folder
    raise FileNotFoundError(f"Could not find a '{wanted}' BUSI layout inside {root}")


layout, base = detect_layout(unzip_dir, LAYOUT)
print(f"Layout: {layout}  (data folder: {base})")

# ---- Pair images with masks ----
MASK_RE = re.compile(r"^(?P<base>.+)_mask(_\d+)?$")      # "x_mask", "x_mask_1", "x_mask_2", ...
CLASS_RE = re.compile(r"^(normal|benign|malignant)", re.IGNORECASE)

if layout == "kaggle":
    files = sorted(p for c in CLASSES for p in (base / c).glob("*.png"))
else:
    files = sorted([*(base / "images").glob("*.png"), *(base / "masks").glob("*.png")])

images, masks = {}, defaultdict(list)
for path in files:
    match = MASK_RE.match(path.stem)
    if match:
        masks[match["base"]].append(path)     # a mask: only used for evaluation, never as input
    else:
        images[path.stem] = path

rows = []
for image_id, path in images.items():
    # Kaggle layout: the class is the folder. Curated layout: the class starts the file name.
    label = path.parent.name.lower() if layout == "kaggle" else CLASS_RE.match(image_id)[1].lower()
    rows.append({"image_id": image_id,
                 "file_name": path.name,
                 "label": label,
                 "image_path": path.as_posix(),
                 "mask_paths": ";".join(p.as_posix() for p in masks[image_id])})
df = pd.DataFrame(rows)
assert not df.image_id.str.contains("_mask").any(), "a mask file slipped into the inputs"


def load_mask(mask_paths, size):
    """Merge all mask files of one image into a single True/False array (empty for normal images)."""
    merged = np.zeros((size[1], size[0]), dtype=bool)
    for p in filter(None, mask_paths.split(";")):
        m = Image.open(p).convert("L")
        if m.size != size:
            m = m.resize(size, Image.NEAREST)
        merged |= np.array(m) > 127
    return merged


n_masks = df.mask_paths.str.count(";") + (df.mask_paths != "")
print(f"\nImages found: {len(df)}")
print(df.label.value_counts().reindex(CLASSES).to_string())
print(f"Images with several masks (merged): {(n_masks > 1).sum()}")
print(f"Tumour images without any mask:     {((n_masks == 0) & (df.label != 'normal')).sum()}")
print(f"Mask files without an image:        {len(set(masks) - set(images))}")

# ---- Near-duplicates with pHash ----
print("\nHashing images for near-duplicate search ...")
hashes = [imagehash.phash(Image.open(p)) for p in df.image_path]
parent = list(range(len(df)))     # union-find: link every near-duplicate pair into one group


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


dup_pairs = []
for i in range(len(df)):
    for j in range(i + 1, len(df)):
        distance = hashes[i] - hashes[j]
        if distance <= PHASH_MAX_DISTANCE:
            dup_pairs.append((df.image_id[i], df.image_id[j], int(distance)))
            parent[find(j)] = find(i)

df["phash"] = [str(h) for h in hashes]
df["dup_group"] = [find(i) for i in range(len(df))]
group_size = df.groupby("dup_group").image_id.transform("size")
dup_groups = df[group_size > 1].groupby("dup_group")

print(f"Near-duplicate pairs (pHash distance <= {PHASH_MAX_DISTANCE}): {len(dup_pairs)}")
print(f"Duplicate groups: {dup_groups.ngroups}, covering {(group_size > 1).sum()} images "
      f"({(group_size > 1).sum() - dup_groups.ngroups} redundant copies)")
for a, b, d in dup_pairs[:30]:
    print(f"  {a}  ~  {b}   (distance {d})")
if len(dup_pairs) > 30:
    print(f"  ... and {len(dup_pairs) - 30} more")
mixed = [dict(zip(g.image_id, g.label)) for _, g in dup_groups if g.label.nunique() > 1]
if mixed:
    print(f"WARNING: {len(mixed)} duplicate group(s) carry different labels:", *mixed, sep="\n  ")

## 2. Split: 70% train / 15% validation / 15% test

Stratified by class, so every split has the same mix of normal / benign / malignant. We split
*duplicate groups*, not single images, so copies of one scan never end up on both sides.

In [ ]:
groups = df.groupby("dup_group").agg(label=("label", "first")).reset_index()
train_g, rest_g = train_test_split(groups, test_size=0.30, stratify=groups.label, random_state=SEED)
# The remaining 30% is cut into two equally random halves. On Curated BUSI (64 normal images) one half
# gets 10 normals and the other 9; the booth needs 10 test images per class, so the test set takes
# the first half (the one with 10).
test_g, val_g = train_test_split(rest_g, test_size=0.50, stratify=rest_g.label, random_state=SEED)

split_of_group = {**dict.fromkeys(train_g.dup_group, "train"),
                  **dict.fromkeys(val_g.dup_group, "val"),
                  **dict.fromkeys(test_g.dup_group, "test")}
df["split"] = df.dup_group.map(split_of_group)
assert df.groupby("dup_group").split.nunique().max() == 1, "a duplicate group crosses splits!"
df.to_csv(OUT_DIR / "split.csv", index=False)

train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)

table = pd.crosstab(df.label, df.split).reindex(index=CLASSES, columns=["train", "val", "test"])
table.loc["total"] = table.sum()
print(table)
print(f"\nSaved {OUT_DIR / 'split.csv'}")

## 3. Train: ImageNet-pretrained ResNet18

* Images resized to 224×224; grey ultrasound copied into 3 channels; ImageNet normalisation.
* Augmentation **for training only**: left-right flip, ±10° rotation, slight brightness/contrast.
  (No up-down flip: in ultrasound the skin is always at the top, so that would create unrealistic images.)
* Class imbalance: mistakes on rare classes count more in the loss (weighted cross-entropy).
* Optimiser AdamW; we keep the epoch with the best **validation macro-F1** (the F1 averaged over the 3 classes).
* A checkpoint is written to Drive **after every epoch**; if one exists, training continues from it.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),       # grey -> 3 identical channels for ResNet
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([                          # validation / test: no randomness
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class BusiDataset(Dataset):
    def __init__(self, frame, transform):
        self.images = [Image.open(p).convert("L") for p in frame.image_path]   # small set: keep in RAM
        self.labels = [CLASSES.index(label) for label in frame.label]
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.transform(self.images[i]), self.labels[i]


train_gen = torch.Generator().manual_seed(SEED)       # controls shuffling and worker seeds
train_loader = DataLoader(BusiDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, generator=train_gen)
val_loader = DataLoader(BusiDataset(val_df, eval_tf), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
test_loader = DataLoader(BusiDataset(test_df, eval_tf), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

# ---- Model, weighted loss, optimiser ----
seed_everything(SEED)
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))     # new 3-class head
model = model.to(device)

counts = train_df.label.value_counts().reindex(CLASSES).to_numpy()
class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32)  # N / (3 * count)
print("Class weights:", dict(zip(CLASSES, class_weights.numpy().round(2))))
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


@torch.no_grad()
def predict(model, loader):
    """Return (probabilities [N x 3], true labels [N]) for a data loader."""
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), dim=1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()


def rng_states():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all(), "loader": train_gen.get_state()}


def restore_rng_states(s):
    random.setstate(s["python"])
    np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch"])
    torch.cuda.set_rng_state_all(s["cuda"])
    train_gen.set_state(s["loader"])


# ---- Resume from Drive if a checkpoint exists ----
CKPT_PATH = CKPT_DIR / "last.pt"               # overwritten after every epoch
BEST_PATH = MODEL_DIR / "busi_resnet18.pt"     # the best epoch by validation macro-F1
split_id = hashlib.md5(",".join(sorted(train_df.image_id)).encode()).hexdigest()   # which data it was trained on

start_epoch, best_f1, history = 1, -1.0, []
if RESUME and CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)   # our own file, so trusted
    if ckpt["split_id"] == split_id and ckpt["epochs"] == EPOCHS:
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        restore_rng_states(ckpt["rng"])
        start_epoch, best_f1, history = ckpt["epoch"] + 1, ckpt["best_f1"], ckpt["history"]
        print(f"RESUMING: epochs 1-{ckpt['epoch']} already done, best val macro-F1 so far {best_f1:.3f}")
    else:
        print("Found a checkpoint from a different split or EPOCHS setting -> training from scratch.")
if start_epoch > EPOCHS:
    print("Training already finished; using the saved model.")

# ---- Training loop ----
for epoch in range(start_epoch, EPOCHS + 1):
    started = time.time()
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    scheduler.step()

    val_probs, val_true = predict(model, val_loader)
    val_f1 = float(f1_score(val_true, val_probs.argmax(1), average="macro"))
    history.append({"epoch": epoch, "train_loss": total_loss / len(train_loader.dataset), "val_macro_f1": val_f1})

    improved = val_f1 > best_f1
    if improved:                                 # keep the best epoch as the final model
        best_f1 = val_f1
        torch.save({"state_dict": model.state_dict(), "classes": CLASSES, "img_size": IMG_SIZE,
                    "epoch": epoch, "val_macro_f1": val_f1}, BEST_PATH)

    # Checkpoint for resuming: write to a temp file first, so a disconnect never leaves a broken file
    tmp = CKPT_PATH.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "rng": rng_states(), "epoch": epoch, "epochs": EPOCHS,
                "best_f1": best_f1, "history": history, "split_id": split_id}, tmp)
    os.replace(tmp, CKPT_PATH)

    print(f"epoch {epoch:2d}/{EPOCHS} | train loss {history[-1]['train_loss']:.3f} | "
          f"val macro-F1 {val_f1:.3f} | {time.time() - started:4.1f}s | checkpoint saved"
          + (" | NEW BEST -> busi_resnet18.pt" if improved else ""))

# ---- Training curves ----
hist = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(hist.epoch, hist.train_loss, color="#C63D76")
ax1.set(title="Training loss", xlabel="epoch")
ax2.plot(hist.epoch, hist.val_macro_f1, color="#243746")
ax2.set(title="Validation macro-F1", xlabel="epoch", ylim=(0, 1))
fig.tight_layout()
fig.savefig(OUT_DIR / "training_curves.png", dpi=150)
plt.show()

## 4. Evaluate on the test set only

The test images were never used for training or for choosing the best epoch, so this is our
honest estimate of how the model does on new images *from the same hospital and machine*.

In [ ]:
checkpoint = torch.load(BEST_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["state_dict"])
print(f"Loaded best model: epoch {checkpoint['epoch']}, val macro-F1 {checkpoint['val_macro_f1']:.3f}\n")

test_probs, y_test = predict(model, test_loader)
y_pred = test_probs.argmax(1)
MAL = CLASSES.index("malignant")

cm = confusion_matrix(y_test, y_pred, labels=range(len(CLASSES)))
report = classification_report(y_test, y_pred, labels=range(len(CLASSES)), target_names=CLASSES,
                               output_dict=True, zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro")
n_malignant = int((y_test == MAL).sum())
caught = int(((y_test == MAL) & (y_pred == MAL)).sum())
sensitivity = caught / n_malignant
false_alarms = int(((y_test != MAL) & (y_pred == MAL)).sum())
auc_macro = roc_auc_score(y_test, test_probs, multi_class="ovr", average="macro", labels=range(len(CLASSES)))
auc_per_class = {c: roc_auc_score(y_test == i, test_probs[:, i]) for i, c in enumerate(CLASSES)}


def wilson_interval(k, n, z=1.96):
    """95% confidence interval for a proportion k/n (honest error bars for small test sets)."""
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return max(0.0, centre - half), min(1.0, centre + half)


sens_low, sens_high = wilson_interval(caught, n_malignant)

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="RdPu", colorbar=False)
ax.set(xlabel="AI prediction", ylabel="True diagnosis", title=f"BUSI test set (n = {len(y_test)})")
fig.tight_layout()
fig.savefig(OUT_DIR / "confusion_matrix.png", dpi=200)
plt.show()

metrics = {
    "n_test": int(len(y_test)),
    "test_counts": {c: int((y_test == i).sum()) for i, c in enumerate(CLASSES)},
    "accuracy": round(float((y_pred == y_test).mean()), 4),
    "macro_f1": round(float(macro_f1), 4),
    "malignant_sensitivity": round(sensitivity, 4),
    "malignant_sensitivity_95ci": [round(sens_low, 4), round(sens_high, 4)],
    "malignant_caught": caught,
    "malignant_missed": n_malignant - caught,
    "false_alarms": false_alarms,
    "per_class": {c: {"precision": round(report[c]["precision"], 4),
                      "recall": round(report[c]["recall"], 4),
                      "f1": round(report[c]["f1-score"], 4),
                      "support": int(report[c]["support"])} for c in CLASSES},
    "roc_auc_ovr_macro": round(float(auc_macro), 4),
    "roc_auc_ovr_per_class": {c: round(float(a), 4) for c, a in auc_per_class.items()},
    "confusion_matrix": {"rows_truth_cols_pred": CLASSES, "matrix": cm.tolist()},
    "best_epoch": int(checkpoint["epoch"]),
    "val_macro_f1": round(checkpoint["val_macro_f1"], 4),
    "dataset_layout": layout,
    "near_duplicate_pairs": len(dup_pairs),
    "split_sizes": {s: int((df.split == s).sum()) for s in ["train", "val", "test"]},
    "seed": SEED,
}
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

print(f"""HONEST SUMMARY
--------------
* Tested on {len(y_test)} ultrasound images the model never saw during training.
* Overall it agreed with the doctors' diagnosis {metrics['accuracy']:.0%} of the time
  (macro-F1 {macro_f1:.2f}; one-vs-rest ROC AUC {auc_macro:.2f}).
* Cancer detection: it flagged {caught} of {n_malignant} malignant tumours (sensitivity {sensitivity:.0%})
  and MISSED {n_malignant - caught}. With so few test cases the true sensitivity could plausibly be
  anywhere from {sens_low:.0%} to {sens_high:.0%} (95% interval).
* It raised {false_alarms} false alarm(s): normal or benign images called malignant.
* Per class: """ + "; ".join(f"{c} precision {report[c]['precision']:.2f} / recall {report[c]['recall']:.2f}"
                             for c in CLASSES) + f"""
* Limits: {len(df)} images from a single hospital and scanner, test set of only {len(y_test)} images.
  Scores like these usually DROP on other hospitals, machines and patient groups.
  This is a teaching demo, not a diagnostic tool.

Saved {OUT_DIR / 'metrics.json'} and confusion_matrix.png""")

## 5. Grad-CAM: where did the model look?

For every test image, Grad-CAM shows which regions pushed the model towards **the class it predicted**
(computed on the last convolutional block, `layer4`). We save:

* **heatmap**: the Grad-CAM overlay on the original image
* **overlay**: the same, plus the doctor-drawn tumour outline in magenta

`cam_mask_overlap` = of the hottest 20% of heatmap pixels, the share inside the true tumour.
Near 1 = it looked at the tumour; near 0 = it looked elsewhere. Empty (null) for normal images.

In [ ]:
(GRADCAM_DIR / "heatmaps").mkdir(parents=True, exist_ok=True)
(GRADCAM_DIR / "overlays").mkdir(parents=True, exist_ok=True)


def overlap_with_mask(cam, mask):
    """Share of the top-20% Grad-CAM pixels that lie inside the tumour mask."""
    k = int(TOP_FRACTION * cam.size)
    hottest = np.argpartition(cam.ravel(), -k)[-k:]      # indices of the k largest values
    return float(mask.ravel()[hottest].mean())


cam_results = {}
model.eval()
with GradCAM(model=model, target_layers=[model.layer4[-1]]) as grad_cam:
    for n, (row, pred) in enumerate(zip(test_df.itertuples(), y_pred), start=1):
        original = Image.open(row.image_path).convert("L")
        x = eval_tf(original).unsqueeze(0).to(device)

        cam = grad_cam(input_tensor=x, targets=[ClassifierOutputTarget(int(pred))])[0]  # 224x224, 0..1
        cam = cv2.resize(cam, original.size, interpolation=cv2.INTER_LINEAR)           # back to full size

        rgb = np.asarray(original.convert("RGB"), dtype=np.float32) / 255
        heatmap = show_cam_on_image(rgb, cam, use_rgb=True)

        mask = load_mask(row.mask_paths, original.size)
        overlay = heatmap.copy()
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(overlay, contours, -1, CONTOUR_RGB, thickness=max(2, original.size[0] // 170))

        heatmap_path = GRADCAM_DIR / "heatmaps" / f"{row.image_id}.png"
        overlay_path = GRADCAM_DIR / "overlays" / f"{row.image_id}.png"
        Image.fromarray(heatmap).save(heatmap_path)
        Image.fromarray(overlay).save(overlay_path)

        has_tumour = row.label != "normal" and mask.any()
        cam_results[row.image_id] = {
            "heatmap_path": heatmap_path.as_posix(),
            "overlay_path": overlay_path.as_posix(),
            "cam_mask_overlap": round(overlap_with_mask(cam, mask), 4) if has_tumour else None,
        }
        if n % 10 == 0 or n == len(test_df):
            print(f"Grad-CAM {n}/{len(test_df)}")

overlaps = pd.Series([r["cam_mask_overlap"] for r in cam_results.values()], dtype=float)
print(f"Median cam_mask_overlap (tumour images): {overlaps.median():.2f}")

## 6. All test predictions

`error_type`: **false_negative** = truly malignant but the AI said otherwise (the dangerous miss);
**false_positive** = normal/benign but the AI said malignant (a false alarm);
**other_confusion** = normal ↔ benign mix-up. Empty when the AI was right.

In [ ]:
def error_type(truth, pred):
    if truth == pred:
        return None
    if truth == "malignant":
        return "false_negative"
    if pred == "malignant":
        return "false_positive"
    return "other_confusion"


pred_df = pd.DataFrame({"image_id": test_df.image_id,
                        "file_name": test_df.file_name,
                        "truth": test_df.label,
                        "pred": [CLASSES[i] for i in y_pred]})
for i, c in enumerate(CLASSES):
    pred_df[f"prob_{c}"] = test_probs[:, i].round(4)
pred_df["confidence"] = test_probs.max(axis=1).round(4)          # probability of the predicted class
pred_df["correct"] = pred_df.truth == pred_df.pred
pred_df["error_type"] = [error_type(t, p) for t, p in zip(pred_df.truth, pred_df.pred)]
pred_df["cam_mask_overlap"] = [cam_results[i]["cam_mask_overlap"] for i in pred_df.image_id]
pred_df["heatmap"] = [cam_results[i]["heatmap_path"] for i in pred_df.image_id]
pred_df["overlay"] = [cam_results[i]["overlay_path"] for i in pred_df.image_id]

pred_df.to_csv(OUT_DIR / "predictions.csv", index=False)
print(pred_df.error_type.fillna("correct").value_counts().to_string())
print(f"\nSaved {len(pred_df)} test predictions to {OUT_DIR / 'predictions.csv'}")

## 7. Booth set: 30 test cases

10 per true class, **test images only**. The set must include:

1. every false negative and false positive available (up to 6, alternating so both kinds appear),
2. the most confident wrong prediction,
3. two "right answer, wrong reason" cases (correct, but the heatmap mostly misses the tumour).

The rest are correct cases spread from low to high confidence. Cases are shuffled, numbered 01–30
and exported for the screen (`images/`, `heatmaps/`, `overlays/`) and for printing (`print/`,
upscaled with a small "Case NN" label and no answer).

In [ ]:
# ---------- Select ----------
rng = np.random.default_rng(SEED)
pool = pred_df.set_index("image_id")            # TEST images only
assert set(pool.index) <= set(df[df.split == "test"].image_id)
chosen = {}                                     # image_id -> list of reasons it was picked


def n_chosen(label):
    return sum(pool.at[i, "truth"] == label for i in chosen)


def pick(image_id, reason, enforce_quota=True):
    """Add a case (or add a reason to a case already picked). Returns True if it is in the set."""
    if image_id in chosen:
        chosen[image_id].append(reason)
        return True
    if enforce_quota and n_chosen(pool.at[image_id, "truth"]) >= BOOTH_PER_CLASS:
        return False
    chosen[image_id] = [reason]
    return True


# 1) False negatives and false positives: most confident first, alternating FN, FP, FN, ...
fn = pool[pool.error_type == "false_negative"].sort_values("confidence", ascending=False).index
fp = pool[pool.error_type == "false_positive"].sort_values("confidence", ascending=False).index
n_errors = 0
for image_id in (i for pair in zip_longest(fn, fp) for i in pair if i is not None):
    if n_errors == MAX_ERRORS:
        break
    n_errors += pick(image_id, pool.at[image_id, "error_type"])

# 2) The most confident wrong prediction (often already in the set from step 1)
wrong = pool[~pool.correct]
if len(wrong):
    pick(wrong.confidence.idxmax(), "most_confident_wrong")

# 3) "Right answer, wrong reason": correct, with the lowest heatmap/tumour overlap
n_low = 0
for image_id in pool[pool.correct & pool.cam_mask_overlap.notna()].sort_values("cam_mask_overlap").index:
    if n_low == N_WRONG_REASON:
        break
    n_low += pick(image_id, "right_answer_wrong_reason")

# 4) Top up every class with correct cases spread over the confidence range
for label in CLASSES:
    need = BOOTH_PER_CLASS - n_chosen(label)
    rest = pool[(pool.truth == label) & ~pool.index.isin(list(chosen))]
    correct = rest[rest.correct].sort_values("confidence")
    if len(correct) > need:                       # evenly spaced from least to most confident
        correct = correct.iloc[np.linspace(0, len(correct) - 1, need).round().astype(int)]
    for image_id in correct.index:
        pick(image_id, "correct_varied_confidence")
    for image_id in rest[~rest.correct].index:    # only used if a class runs out of correct cases
        pick(image_id, "extra_error")

# 5) Safety net: if a class has fewer than 10 test images, fill up with other TEST images
n_cases = BOOTH_PER_CLASS * len(CLASSES)
if len(chosen) < n_cases:
    shortfall = n_cases - len(chosen)
    print(f"WARNING: a class has fewer than {BOOTH_PER_CLASS} test images; adding {shortfall} other test images.")
    leftovers = pool.index[~pool.index.isin(list(chosen))]
    for image_id in rng.choice(leftovers, size=shortfall, replace=False):
        pick(image_id, "fill_shortage", enforce_quota=False)

order = list(chosen)
rng.shuffle(order)
booth = pool.loc[order].reset_index()
booth.insert(0, "case_id", [f"{k:02d}" for k in range(1, len(booth) + 1)])
booth["why_selected"] = [" + ".join(chosen[i]) for i in booth.image_id]

print(f"Booth cases: {len(booth)} -> " + ", ".join(f"{c} {n}" for c, n in booth.truth.value_counts()
                                                     .reindex(CLASSES).items()))
print(f"False negatives available {len(fn)}, false positives available {len(fp)}, "
      f"included {n_errors} (cap {MAX_ERRORS})")


# ---------- Export ----------
def load_font(size):
    for name in ("DejaVuSans-Bold.ttf", "arialbd.ttf", "Arial Bold.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    return ImageFont.load_default(size=size)       # Pillow >= 10.1


def make_print(image, case_id):
    """Upscale for printing and add a small 'Case NN' label. The answer is NOT shown."""
    image = image.convert("RGB")
    height = round(image.height * PRINT_WIDTH / image.width)
    big = image.resize((PRINT_WIDTH, height), Image.LANCZOS)
    draw = ImageDraw.Draw(big, "RGBA")
    font = load_font(PRINT_WIDTH // 32)
    margin = PRINT_WIDTH // 20                    # inset ~7 mm so a mat / frame edge won't cover the label
    pad = PRINT_WIDTH // 120
    text = f"Case {case_id}"
    box = draw.textbbox((margin, margin), text, font=font)
    draw.rounded_rectangle((box[0] - pad, box[1] - pad, box[2] + pad, box[3] + pad),
                           radius=pad, fill=(255, 255, 255, 215))
    draw.text((margin, margin), text, font=font, fill=(36, 55, 70))
    return big


for sub in ("images", "heatmaps", "overlays", "print"):
    (EXPORT_DIR / sub).mkdir(parents=True, exist_ok=True)

image_paths = df.set_index("image_id").image_path
booth_json = []
for case in booth.itertuples():
    cid = case.case_id
    original = Image.open(image_paths[case.image_id]).convert("L")
    original.save(EXPORT_DIR / f"images/case{cid}.png")
    make_print(original, cid).save(EXPORT_DIR / f"print/case{cid}_A5.png", dpi=(300, 300))
    shutil.copyfile(case.heatmap, EXPORT_DIR / f"heatmaps/case{cid}.png")
    shutil.copyfile(case.overlay, EXPORT_DIR / f"overlays/case{cid}.png")

    booth_json.append({
        "case_id": cid,
        "source_file": case.file_name,
        "image": f"images/case{cid}.png",
        "print_image": f"print/case{cid}_A5.png",
        "heatmap": f"heatmaps/case{cid}.png",
        "mask_overlay": f"overlays/case{cid}.png",
        "truth": case.truth,
        "ai_pred": case.pred,
        "ai_probs": {c: round(float(getattr(case, f"prob_{c}")), 4) for c in CLASSES},
        "correct": bool(case.correct),
        "error_type": None if pd.isna(case.error_type) else case.error_type,
        "cam_mask_overlap": None if pd.isna(case.cam_mask_overlap) else round(float(case.cam_mask_overlap), 3),
    })

# allow_nan=False: fail loudly rather than write invalid JSON (NaN) for the booth app
(EXPORT_DIR / "booth_set.json").write_text(json.dumps(booth_json, indent=2, allow_nan=False))

# Staff answer key: why each case was chosen (keep away from visitors!)
booth.sort_values("case_id")[["case_id", "file_name", "truth", "pred", "confidence", "error_type",
                              "cam_mask_overlap", "why_selected"]].to_csv(EXPORT_DIR / "booth_key.csv", index=False)

# Every test error, so you can choose cases for the printed "patient folders" by hand
failures = pred_df[~pred_df.correct].copy()
failures["booth_case_id"] = failures.image_id.map(dict(zip(booth.image_id, booth.case_id)))
failures["severity"] = failures.error_type.map({"false_negative": 0, "false_positive": 1, "other_confusion": 2})
failures = failures.sort_values(["severity", "confidence"], ascending=[True, False]).drop(columns="severity")
failures.to_csv(EXPORT_DIR / "failure_candidates.csv", index=False)

print(f"Exported {len(booth_json)} cases and {len(failures)} failure candidates to {EXPORT_DIR}")

# ---------- Quick look at the teaching moments ----------
special = booth[booth.why_selected != "correct_varied_confidence"].sort_values("case_id")
fig, axes = plt.subplots(len(special), 3, figsize=(9, 3 * len(special)), squeeze=False)
for row_axes, case in zip(axes, special.itertuples()):
    for ax, folder in zip(row_axes, ["images", "heatmaps", "overlays"]):
        ax.imshow(Image.open(EXPORT_DIR / f"{folder}/case{case.case_id}.png"), cmap="gray")
        ax.axis("off")
    overlap = "n/a" if pd.isna(case.cam_mask_overlap) else f"{case.cam_mask_overlap:.2f}"
    row_axes[0].set_title(f"Case {case.case_id}: truth {case.truth}", fontsize=9)
    row_axes[1].set_title(f"AI: {case.pred} ({case.confidence:.0%})", fontsize=9)
    row_axes[2].set_title(f"{case.why_selected}\noverlap {overlap}", fontsize=8)
fig.tight_layout()
plt.show()

## 8. Wisconsin (WDBC) model

A tiny model on five numeric tumour measurements, for the "Tweak the Tumor" booth page:
`StandardScaler` → `LogisticRegression`, stratified 80/20 split.

**Watch out:** in scikit-learn's copy of this dataset, `0 = malignant` and `1 = benign`. We turn the
numbers into the words "malignant" / "benign" right away, so nothing downstream can mix them up.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.frame[WDBC_FEATURES].to_numpy()
y = data.target_names[data.target]        # target_names == ['malignant', 'benign'], so 0 -> 'malignant'
assert (y == "malignant").sum() == 212 and (y == "benign").sum() == 357, "unexpected label mapping"

X_train, X_test, yw_train, yw_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
wdbc_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))
wdbc_model.fit(X_train, yw_train)

yw_pred = wdbc_model.predict(X_test)
wdbc_cm = confusion_matrix(yw_test, yw_pred, labels=["benign", "malignant"])
print(f"Test samples: {len(yw_test)} ({(yw_test == 'malignant').sum()} malignant)")
print(f"Accuracy:         {accuracy_score(yw_test, yw_pred):.3f}")
print(f"Malignant recall: {recall_score(yw_test, yw_pred, pos_label='malignant'):.3f}  "
      f"(caught {wdbc_cm[1, 1]} of {wdbc_cm[1].sum()} malignant tumours, missed {wdbc_cm[1, 0]})")

# Which way does each feature push? (positive = towards malignant, per standard deviation)
coefs = wdbc_model[-1].coef_[0] * (1 if wdbc_model.classes_[1] == "malignant" else -1)
print("\nEffect of +1 standard deviation (log-odds of malignant):")
for name, c in sorted(zip(WDBC_FEATURES, coefs), key=lambda t: -abs(t[1])):
    print(f"  {name:22s} {c:+.2f}")

with open(MODEL_DIR / "wdbc_model.pkl", "wb") as f:
    pickle.dump(wdbc_model, f)
feature_info = [{"name": name, "min": float(X[:, i].min()), "max": float(X[:, i].max()),
                 "median": float(np.median(X[:, i]))} for i, name in enumerate(WDBC_FEATURES)]
(MODEL_DIR / "wdbc_features.json").write_text(json.dumps(feature_info, indent=2))
print(f"\nSaved {MODEL_DIR / 'wdbc_model.pkl'} and wdbc_features.json")
print("\nAll done. Everything is in", OUT_DIR)